# Date Standardisation

## Overview
Date and time data arrives in dozens of formats. Standardising to ISO 8601 (`YYYY-MM-DD`) is the single most important date cleaning step — it makes dates sortable as strings, comparable, and compatible with all date functions.

**Common input formats encountered:**
- `2023-03-15` — ISO 8601 ✓ (already standard)
- `15/03/2023` — DD/MM/YYYY (common in UK/Canada/Europe)
- `03/15/2023` — MM/DD/YYYY (US format)
- `15-03-2023` — DD-MM-YYYY (European with dashes)
- `Jan 30 1955` — Month name format
- `20230315` — Compact ISO

**SQLite date functions:**
- `DATE(date_string)` — parses ISO dates; returns NULL for non-ISO
- `STRFTIME(format, date_string)` — format and reformat dates
- `JULIANDAY(date_string)` — convert to Julian day number for arithmetic
- `DATE(date_string, '+N days')` — date arithmetic

**PostgreSQL date functions:** `TO_DATE(str, format)`, `TO_CHAR(date, format)`, `DATE_PART()`, `AGE()`, `date_trunc()`

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE intake_raw (record_id INTEGER PRIMARY KEY, patient_ref TEXT, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, phone TEXT, email TEXT, cost_str TEXT, intake_date TEXT, notes TEXT);
CREATE TABLE lab_raw (lab_id INTEGER PRIMARY KEY, patient_ref TEXT, test_code TEXT, result_txt TEXT, collected TEXT, status TEXT);
CREATE TABLE provider_raw (prov_id INTEGER PRIMARY KEY, name_raw TEXT, dept_code TEXT, start_dt TEXT, salary TEXT);
INSERT INTO intake_raw VALUES
  (1,'P-001','aroha','Ngata','1985-03-12','F','NB','506-555-0101','aroha@mail.com','$3,200.00','2023-01-15','Referred by GP'),
  (2,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (3,'P-003','FATIMA','AL-RASHID','1990-07-22','female','Ontario','416-555-0303',NULL,'120.00','2023-03-05','Annual checkup'),
  (4,'P-004','James','MacLeod','Jan 30 1955','M','NB',NULL,'james@mail.com','$5,500','18-03-2023','Surgery follow-up'),
  (5,'P-005','sofia','Petrov','2001-09-15','F','BC','604-555-0505','sofia@mail.com','$95.00','2023-04-02',NULL),
  (6,'P-006','Noah','Williams','08/06/1968','m','AB ','780-555-0606','noah@mail.com','780','11/05/2023',NULL),
  (7,'P-007','Mei','Zhang','1995-04-17','F','ON','416-555-0707',NULL,'$2,100.00','22-06-2023','Follow-up required'),
  (8,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (9,'P-008','ethan','tremblay','01/12/1980',NULL,'QC','514-555-0808','ethan@mail.com','80.00','14-07-2023',NULL),
  (10,'P-009','Priya','Nair','1977-08-25','F','bc',NULL,'priya@mail.com','$1,750.00','01/10/2023',NULL),
  (11,'P-010','Marcus','Okafor','1993-05-19','M','ON','647-555-1010',NULL,'2200','03-11-2023',NULL),
  (12,'P-011','Diana','Flores','14/02/2000','female','NB','506-555-1111','diana@mail.com',NULL,'2023-12-01',NULL),
  (13,NULL,'Unknown',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,'Incomplete record'),
  (14,'','Test','Record','2023-01-01','X','XX','000-000-0000','test@test.com','-1','2023-01-01','Test entry');
INSERT INTO lab_raw VALUES
  (1,'P-001','HBA1C','7.2 %','2023-03-10','Active'),
  (2,'P-002','HBA1C','9.1%','2023-03-15','active'),
  (3,'P-003','CREAT','88.5umol/L','2023-04-01','ACTIVE'),
  (4,'P-004','CREAT','145 umol/L','2023-04-12','Active'),
  (5,'P-005','HBA1C','5.8 %','2023-05-05',''),
  (6,'P-006','SODIUM','138mmol/L','2023-05-20','Active'),
  (7,'P-007','SODIUM','151 mmol/L','2023-06-01',NULL),
  (8,'P-001','CREAT','72.3 umol/L','2023-06-18','Active'),
  (9,'P-008','HBA1C','6.4%','2023-07-02','Active'),
  (10,'P-009','CREAT','210umol/L','2023-07-15','active');
INSERT INTO provider_raw VALUES
  (1,'MacNeil, Sarah MD','CARD','2018-01-15','$95,000'),
  (2,'Dr. James Wong','ONCO','2019-03-01','88000'),
  (3,'Fatima Osei M.D.','FAM','2017-06-01','$82,500.00'),
  (4,'Larson, Ethan','ORTH','2020-09-15','91000.00'),
  (5,'Sharma, Priya MD','EMRG','2016-11-01','$78,000'),
  (6,'Lucas Petit','RAD','2021-02-28',NULL);
""")
conn.commit()
print("Schema ready. Date formats in intake_raw:")
print(pd.read_sql("SELECT record_id, dob, intake_date FROM intake_raw ORDER BY record_id", conn).to_string(index=False))

---
## Detecting date format and validating ISO dates

In [ ]:
print("=== Identify which dates are valid ISO 8601 (YYYY-MM-DD) ===")
q = """
SELECT  record_id,
        dob,
        intake_date,
        -- DATE() returns NULL for non-ISO strings in SQLite
        DATE(dob)           AS dob_parsed,
        DATE(intake_date)   AS intake_parsed,
        CASE WHEN DATE(dob) IS NULL AND dob IS NOT NULL
             THEN 'needs conversion' ELSE 'ok or null' END AS dob_status,
        CASE WHEN DATE(intake_date) IS NULL AND intake_date IS NOT NULL
             THEN 'needs conversion' ELSE 'ok or null' END AS intake_status
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

---
## Converting common non-ISO formats to YYYY-MM-DD

In [ ]:
# Identify format by pattern, then rearrange parts
print("=== Format detection and normalisation ===")
q = """
SELECT  record_id,
        dob,
        -- Detect and convert known formats:
        CASE
            -- Already ISO: YYYY-MM-DD (length 10, dash at pos 5 and 8)
            WHEN dob GLOB '????-??-??' THEN dob

            -- DD/MM/YYYY: swap to YYYY-MM-DD
            WHEN dob GLOB '??/??/????' THEN
                SUBSTR(dob,7,4)||'-'||SUBSTR(dob,4,2)||'-'||SUBSTR(dob,1,2)

            -- DD-MM-YYYY
            WHEN dob GLOB '??-??-????' THEN
                SUBSTR(dob,7,4)||'-'||SUBSTR(dob,4,2)||'-'||SUBSTR(dob,1,2)

            ELSE NULL  -- 'Jan 30 1955' needs app-layer parsing or PostgreSQL TO_DATE
        END  AS dob_iso,
        intake_date,
        CASE
            WHEN intake_date GLOB '????-??-??' THEN intake_date
            WHEN intake_date GLOB '??/??/????' THEN
                SUBSTR(intake_date,7,4)||'-'||SUBSTR(intake_date,4,2)||'-'||SUBSTR(intake_date,1,2)
            WHEN intake_date GLOB '??-??-????' THEN
                SUBSTR(intake_date,7,4)||'-'||SUBSTR(intake_date,4,2)||'-'||SUBSTR(intake_date,1,2)
            ELSE NULL
        END  AS intake_iso
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

---
## STRFTIME — reformatting and extracting date parts

In [ ]:
print("=== STRFTIME: extract year, month, day; reformat ===")
q = """
SELECT  record_id,
        DATE(intake_date)                           AS intake_iso,
        STRFTIME('%Y', DATE(intake_date))          AS year,
        STRFTIME('%m', DATE(intake_date))          AS month,
        STRFTIME('%d', DATE(intake_date))          AS day,
        STRFTIME('%Y-%m', DATE(intake_date))       AS year_month,
        STRFTIME('%d/%m/%Y', DATE(intake_date))    AS dd_mm_yyyy
FROM    intake_raw
WHERE   intake_date GLOB '????-??-??'   -- only already-ISO rows
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Age calculation using JULIANDAY or SQLite date arithmetic
print()
print("=== Age in years from dob ===")
q2 = """
SELECT  record_id,
        first_name,
        dob,
        CAST((JULIANDAY('2023-12-31') - JULIANDAY(dob)) / 365.25 AS INTEGER) AS age_years
FROM    intake_raw
WHERE   dob GLOB '????-??-??'
ORDER BY age_years DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()

---
## Common Pitfalls

**1. SQLite's DATE() silently returns NULL for non-ISO strings**
`DATE('15/03/2023')` returns NULL — no error, no warning. Always validate with `CASE WHEN DATE(col) IS NULL AND col IS NOT NULL THEN 'bad format'` before using date functions.

**2. Ambiguous two-digit-first formats: DD/MM or MM/DD?**
`04/11/1972` could be November 4 or April 11 depending on locale. You cannot resolve this from the data alone without domain context or locale metadata. Document the assumption and flag ambiguous dates for review.

**3. String-sorting ISO dates works; sorting non-ISO strings doesn't**
`ORDER BY dob` where `dob` is ISO `YYYY-MM-DD` sorts correctly because ISO string order equals chronological order. `ORDER BY '15/03/2023'` sorts as a string (wrong). Always convert to ISO before sorting or comparing.

**4. STRFTIME('%Y', col) returns NULL if col is not a valid SQLite date**
Even after conversion attempts, some dates may still be non-ISO. Add `WHERE DATE(col) IS NOT NULL` before applying STRFTIME to avoid NULL propagation in downstream columns.

**5. Age calculation with JULIANDAY is approximate**
`(JULIANDAY('2024-01-01') - JULIANDAY(dob)) / 365.25` approximates age in years. It doesn't account for leap years precisely. For exact age, use PostgreSQL's `AGE()` function or compare year/month/day parts explicitly.

**6. PostgreSQL TO_DATE() with the wrong format mask silently produces wrong dates**
`TO_DATE('04/11/1972', 'DD/MM/YYYY')` = 1972-11-04. `TO_DATE('04/11/1972', 'MM/DD/YYYY')` = 1972-04-11. The function doesn't validate which format is correct — it applies whatever mask you give it. Always verify a sample of converted dates against source records.


---
*sql_methods_library - Samantha McGarrigle*